## Imports

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import matplotlib.pyplot as plt
import os
import pandas as pd
import torch
from torch import optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm 
import seaborn as sns

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU detected: NVIDIA GeForce RTX 3060 Laptop GPU


## Data Loading

In [ ]:
path = "../../data"
print("Path to dataset files : ",path)
# Paths to the CSV files
train_csv_path = os.path.join(path, 'train.csv')
test_csv_path = os.path.join(path, 'test.csv')

image_dir = os.path.join(path, 'faces')

df_train = pd.read_csv(train_csv_path)
df_train['id'] = df_train['id'].astype(int).astype(str)
df_train['id'] = 'face-'+df_train['id']+'.png'

df_test = pd.read_csv(test_csv_path)
df_test['id'] = df_test['id'].astype(int).astype(str)
df_test['id'] = 'face-'+df_test['id']+'.png'

print(f"Total images found in train.csv: {len(df_train)}")
print(f"Total images found in train.csv: {len(df_test)}")

Path to dataset files :  ./data
Total images found in train.csv: 4500
Total images found in train.csv: 500


## Image Downscaling

In [4]:
final_dim = 64
processed_dir = path + "/faces_processed_" + str(final_dim)
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)


# Define your interval (e.g., every 50 images)
progress_interval = 50 
count = 0

# Correcting the concat syntax: it needs a list [df_train, df_test]
combined_df = pd.concat([df_train, df_test])
total_images = len(combined_df)

print(f"Resizing images to {final_dim}x{final_dim}...")

for index, row in combined_df.iterrows():
    filename = row['id']
    input_path = os.path.join(image_dir, filename)
    output_path = os.path.join(processed_dir, filename)
    
    # 1. Check if the file already exists in the output directory
    if os.path.exists(output_path):
        count += 1
        continue # Skip to the next image
    
    img = cv2.imread(input_path)
    
    if img is not None:
        resized_img = cv2.resize(img, (final_dim, final_dim), interpolation=cv2.INTER_AREA)
        cv2.imwrite(output_path, resized_img)
        count += 1
        
        # 2. Progress indicator with custom counter
        if count % progress_interval == 0 or count == total_images:
            print(f"Progress: {count}/{total_images} images processed.")
    else:
        print(f"Warning: Could not read {filename} at {input_path}")
        count += 1

print(f"Successfully saved resized images to {processed_dir}")

Resizing images to 64x64...
Successfully saved resized images to ./data/faces_processed_64
